In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from inspect import Parameter
import math
import networkx as nx
#from dynamic_similarities import build_dynamic_similarity_graphs, save_dynamic_graph_plots
#from lstm_dataset import TimeSeriesDataset


In [ ]:
def build_statistical_similarity_graph(
    df: pd.DataFrame,
    date_col: str,
    item_col: str,
    target_col: str,
    aggfunc: str = "sum",
    similarity_method: str = "pearson",   # "pearson", "spearman", "kendall"
    similarity_threshold: float = 0.7,
    k: int = None,
    use_absolute_similarity: bool = False,
):
    # 1) Pivot: rows = dates, cols = items
    df_pivot = (
        df.pivot_table(
            index=date_col,
            columns=item_col,
            values=target_col,
            aggfunc=aggfunc
        )
        .sort_index()
        .ffill()
        .fillna(0)
    )

    # 2) Compute item-item similarity matrix
    # Correlate columns (items) across time
    sim_df = df_pivot.corr(method=similarity_method)

    item_ids = sim_df.columns.tolist()

    # 3) Build graph
    G = nx.Graph(name=f"{similarity_method.capitalize()}_Similarity_Graph")
    G.add_nodes_from(item_ids)

    n = len(item_ids)

    if k is not None:
        # k-NN graph: connect each node to top-k most similar neighbors
        for i in range(n):
            sim_row = sim_df.iloc[i].copy()
            sim_row.iloc[i] = np.nan  # remove self-correlation

            if use_absolute_similarity:
                top_k_neighbors = sim_row.abs().nlargest(k).dropna()
            else:
                top_k_neighbors = sim_row.nlargest(k).dropna()

            for neighbor_id, sim_value in top_k_neighbors.items():
                j = sim_df.columns.get_loc(neighbor_id)

                edge_weight = abs(sim_value) if use_absolute_similarity else sim_value

                if use_absolute_similarity or sim_value >= similarity_threshold:
                    G.add_edge(
                        item_ids[i],
                        neighbor_id,
                        weight=float(edge_weight),
                        similarity=float(sim_value)
                    )
                    print(
                        f"Added edge between {item_ids[i]} and {neighbor_id} "
                        f"with {similarity_method} similarity: {sim_value:.4f}"
                    )

    else:
        # Threshold graph
        for i in range(n):
            for j in range(i + 1, n):
                sim_value = sim_df.iloc[i, j]

                if pd.isna(sim_value):
                    continue

                edge_weight = abs(sim_value) if use_absolute_similarity else sim_value

                condition = (
                    abs(sim_value) >= similarity_threshold
                    if use_absolute_similarity
                    else sim_value >= similarity_threshold
                )

                if condition:
                    G.add_edge(
                        item_ids[i],
                        item_ids[j],
                        weight=float(edge_weight),
                        similarity=float(sim_value)
                    )
                    print(
                        f"Added edge between {item_ids[i]} and {item_ids[j]} "
                        f"with {similarity_method} similarity: {sim_value:.4f}"
                    )

    print(f"Number of nodes in the {similarity_method} graph:", G.number_of_nodes())
    print(f"Number of edges in the {similarity_method} graph:", G.number_of_edges())

    return G, sim_df, df_pivot

In [ ]:
def build_dynamic_similarity_graphs(
    df: pd.DataFrame,
    date_col: str,
    item_col: str,
    target_col: str,
    window_size: int,
    step_size: int,
    aggfunc: str = "sum",
    similarity_method: str = "pearson",
    similarity_threshold: float = 0.7,
    k: int = None,
    use_absolute_similarity: bool = False,
):
    """
    Builds a sequence of dynamic similarity graphs using a sliding window 
    over the dates in the dataframe.
    """
    unique_dates = sorted(df[date_col].unique())
    
    num_dates = len(unique_dates)
    graphs = []
    sim_dfs = []
    df_pivots = []
    window_info = []

    for start_idx in range(0, num_dates - window_size + 1, step_size):
        end_idx = start_idx + window_size
        current_dates = unique_dates[start_idx:end_idx]
        
        start_date = current_dates[0]
        end_date = current_dates[-1]
        
        mask = df[date_col].isin(current_dates)
        df_window = df[mask]
        
        print(f"\nBuilding graph for window: {start_date} to {end_date}")
        
        G, sim_df, df_pivot = build_statistical_similarity_graph(
            df=df_window,
            date_col=date_col,
            item_col=item_col,
            target_col=target_col,
            aggfunc=aggfunc,
            similarity_method=similarity_method,
            similarity_threshold=similarity_threshold,
            k=k,
            use_absolute_similarity=use_absolute_similarity
        )
        
        G.graph["start_date"] = start_date
        G.graph["end_date"] = end_date
        
        graphs.append(G)
        sim_dfs.append(sim_df)
        df_pivots.append(df_pivot)
        window_info.append({"start_date": start_date, "end_date": end_date})
        
    return graphs, sim_dfs, df_pivots, window_info

In [ ]:
DATA_PATH = '../../dataset/representative_items.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

NUM_ITEMS = 100
df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

In [ ]:
# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)
ITEM_COL = 'item_id'

# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df['item_id'] = df['item_id'].astype(int) # or .astype(str)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)

m0 = df[DATE_COL].dt.month - 1
df["month_sin"] = np.sin(2*np.pi*m0 / 12)
df["month_cos"] = np.cos(2*np.pi*m0 / 12)

df["doy"] = df[DATE_COL].dt.dayofyear # 1..365/366
doy0 = df["doy"] - 1
P = 366 # safe; or use 365 if you drop leap years
df["doy_sin"] = np.sin(2*np.pi*doy0 / P)
df["doy_cos"] = np.cos(2*np.pi*doy0 / P)

promo_types= [col for col in df.columns if col.startswith("promo_type_")] # Binary columns indicating presence of specific promotion types
promo_values= [col for col in df.columns if col.startswith("promo_value_")] # Numerical columns indicating the value of specific promotion types

calendar_cols = ["day_of_week", "day_of_month", "moy", "doy", "is_weekend"]
calendar_trigonometric_cols = ["month_sin", "month_cos", "doy_sin", "doy_cos"]
categorical_cols = ["cat_label", "sdep_label", "dept_label"]

# USE THE EXOGENOUS COLUMNS YOU CREATED
EXOG_COLS = ["day_of_week", "doy", "is_thanksgiving", "is_christmas","is_weekend"]

df = df.sort_values(["item_id", DATE_COL]).reset_index(drop=True)

df

# Graph Construction setup

In [ ]:
WINDOW_SIZE = 56
STEP_SIZE = 7
SIMILARITY_METHOD = "kendall"
SIMILARITY_THRESHOLD = 0.8

In [ ]:
graphs, sim_dfs, df_pivots, window_info = build_dynamic_similarity_graphs(
    df,
    date_col=DATE_COL,
    item_col=ITEM_COL,
    target_col=TARGET_COL,
    window_size=WINDOW_SIZE,
    step_size=STEP_SIZE,
    similarity_method=SIMILARITY_METHOD,
    similarity_threshold=SIMILARITY_THRESHOLD
)

# GCN Model Definition

In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolution, self).__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(torch.FloatTensor(in_features, out_features))

        if bias:
            self.bias = Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))        
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, input, adj):
        support = torch.mm(input, self.weight)
        output = torch.spmm(adj, support)
        if self.bias is not None:
            return output + self.bias
        else:
            return output

    def __repr__(self):
        return f'{self.__class__.__name__} ({self.in_features} -> {self.out_features})'


class GCN(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout):
        super(GCN, self).__init__()

        self.gc1 = GraphConvolution(nfeat, nhid)
        self.gc2 = GraphConvolution(nhid, nclass)
        self.dropout = dropout

    def forward(self, x, adj):
        x = F.relu(self.gc1(x, adj))
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj)
        return x
        #return F.log_softmax(x, dim=1)

In [ ]:
gcn_model = GCN(input_dim=16, hidden_dim=32, output_dim=16)  # Example dimensions
seq_length = 28

# LSTM Setup

In [ ]:
seq_length = 28

In [ ]:
def compute_window_node_features(df_pivot: pd.DataFrame):
    """
    Computes node features for a single graph window, shape: (num_items, num_features)
    """
    num_items = df_pivot.shape[1]
    window_size = df_pivot.shape[0]
    
    features = []
    
    # Iterate over items
    for j in range(num_items):
        item_ts = df_pivot.iloc[:, j].values
        
        # Base stats
        last_demand = item_ts[-1] if window_size > 0 else 0
        
        # Rolling means
        mean7 = np.mean(item_ts[-7:]) if window_size >= 7 else np.mean(item_ts)
        mean28 = np.mean(item_ts[-28:]) if window_size >= 28 else np.mean(item_ts)
        
        # Std dev
        std28 = np.std(item_ts[-28:]) if window_size >= 28 else np.std(item_ts)
        
        # Zero-demand ratio
        if window_size >= 28:
            zero_ratio28 = np.mean(item_ts[-28:] == 0)
        else:
            zero_ratio28 = np.mean(item_ts == 0)
            
        # Slope/trend over last 28 days
        if window_size >= 28:
            y_28 = item_ts[-28:]
            x_28 = np.arange(28)
            # using polyfit to get slope
            slope28 = np.polyfit(x_28, y_28, 1)[0]
        elif window_size > 1:
            slope28 = np.polyfit(np.arange(window_size), item_ts, 1)[0]
        else:
            slope28 = 0.0
            
        min_28 = np.min(item_ts[-28:]) if window_size >= 28 else np.min(item_ts)
        max_28 = np.max(item_ts[-28:]) if window_size >= 28 else np.max(item_ts)
            
        features.append([
            last_demand,
            mean7,
            mean28,
            std28,
            zero_ratio28,
            slope28,
            min_28,
            max_28
        ])
        
    return np.array(features)

# Apply this to loop through all dynamic graph windows
dynamic_graph_features = []

for pivot_table in df_pivots:
    # Compute numpy array of features
    feats = compute_window_node_features(pivot_table)
    
    # Convert directly to torch float tensor
    dynamic_graph_features.append(torch.tensor(feats, dtype=torch.float32))

# Here they are ready to be utilized iteratively inside your training loop!
# Checking shape of the first window:
print(f"Features dimension for window 1: {dynamic_graph_features[0].shape}") 
# Expected: (num_items, 8)

In [ ]:
class DynamicGraphTimeSeriesDataset(Dataset):
    def __init__(
        self, 
        target_data, 
        exog_data, 
        date_data,
        seq_length, 
        dynamic_graphs, 
        dynamic_features, 
        graph_window_info
    ):
        """
        Dataset for time series that intelligently fetches the most recent valid graph context.
        
        Args:
            target_data: Target variable data (e.g. sales)
            exog_data: Exogenous variables data (can be None)
            date_data: Array or series of dates corresponding to each time step
            seq_length: Length of LSTM input sequences (e.g. 28)
            dynamic_graphs: List of adjacency matrices/edge_indices for each graph window
            dynamic_features: List of computed node feature tensors for each graph window
            graph_window_info: List of dicts with 'start_date' and 'end_date' for each window
        """
        self.target_data = target_data
        self.exog_data = exog_data
        self.date_data = pd.to_datetime(date_data)  # Ensure datetime format for exact comparisons
        self.seq_length = seq_length
        self.has_exog = exog_data is not None
        
        self.dynamic_graphs = dynamic_graphs
        self.dynamic_features = dynamic_features
        self.graph_window_info = graph_window_info
        
        # Pre-process window dates for rapid lookup during __getitem__
        self.window_end_dates = pd.to_datetime([info['end_date'] for info in graph_window_info])
        
    def __len__(self):
        return len(self.target_data) - self.seq_length
    
    def _find_latest_valid_graph_index(self, target_date):
        """
        Finds the index of the most recent graph that was completed on or before `target_date`.
        """
        # Find all windows that ended before or on the current target date
        valid_indices = np.where(self.window_end_dates <= target_date)[0]
        
        if len(valid_indices) == 0:
            # If we ask for a date so early that NO graph window has finished yet,
            # fallback to the very first available graph (or handle as 0 padding)
            return 0
            
        # Return the index of the most recently finished window
        return valid_indices[-1]
    
    def __getitem__(self, idx):
        # 1. Temporal sequence preparation (Standard LSTM logic)
        target_seq = self.target_data[idx : idx + self.seq_length]
        
        # The exact day we are predicting sales for
        forecast_horizon_idx = idx + self.seq_length
        y = self.target_data[forecast_horizon_idx]
        
        # The last day the LSTM has access to (the day before the prediction)
        last_observed_date = self.date_data.iloc[forecast_horizon_idx - 1] if isinstance(self.date_data, pd.Series) else self.date_data[forecast_horizon_idx - 1]
        
        # Compile temporal features X
        if self.has_exog:
            exog_seq = self.exog_data[idx + 1 : forecast_horizon_idx + 1] 
            x_ts = np.column_stack([target_seq.reshape(-1, 1), exog_seq])
        else:
            x_ts = target_seq.reshape(-1, 1)
            
        # 2. Structural Dynamic Graph preparation
        # Find which graph snapshot was structurally valid as of `last_observed_date`
        graph_idx = self._find_latest_valid_graph_index(last_observed_date)
        
        # Extract the correctly synched graph matrices
        graph_adj = self.dynamic_graphs[graph_idx]
        graph_x = self.dynamic_features[graph_idx]
        
        return (
            torch.FloatTensor(x_ts), 
            torch.FloatTensor([y]),
            graph_adj,  # The adjacency structure valid for this exact timestamp
            graph_x     # The node features valid for this exact timestamp
        )


In [ ]:
class DynamicGraphEncoder(nn.Module):
    def __init__(self, gcn_model):
        """
        A dedicated graph encoder that takes a sequence of graphs and returns their dynamic embeddings.
        """
        super(DynamicGraphEncoder, self).__init__()
        self.gcn = gcn_model

    def forward(self, graph_x_seq, graph_adj_seq, target_node_idx=None):
        """
        Encodes a sequence of graphs natively.
        If target_node_idx is provided, extracts embeddings only for that node.
        Otherwise, returns embeddings for all nodes.
        
        Outputs tensor shape:
        - If target_node_idx is not None: (seq_length, gcn_embed_dim)
        - If target_node_idx is None: (seq_length, num_nodes, gcn_embed_dim)
        """
        seq_length = len(graph_x_seq)
        dynamic_embeddings = []
        
        for t in range(seq_length):
            h_t = F.relu(self.gcn.gc1(graph_x_seq[t], graph_adj_seq[t]))
            node_embeddings_t = self.gcn.gc2(h_t, graph_adj_seq[t])
            
            if target_node_idx is not None:
                z_i_t = node_embeddings_t[target_node_idx]
                dynamic_embeddings.append(z_i_t)
            else:
                dynamic_embeddings.append(node_embeddings_t)
                
        # Stack embeddings along the sequence dimension
        return torch.stack(dynamic_embeddings, dim=0)

In [ ]:
class LateFusion_Discrete_GCN_LSTM(nn.Module):
    def __init__(self, gcn_model, lstm_input_size, lstm_hidden_size, lstm_num_layers, gcn_embed_dim, horizon=1, dropout=0.2):
        super(LateFusion_Discrete_GCN_LSTM, self).__init__()
        
        self.gcn = gcn_model
        
        # LSTM processes ONLY the temporal features (no graph concatenation yet)
        self.lstm = nn.LSTM(
            input_size=lstm_input_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True,
            dropout=dropout if lstm_num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        
        # MLP takes the combined [h_ts || z_graph] to predict the horizon
        # Combined size is lstm_hidden_size (from last LSTM output) + gcn_embed_dim
        self.fc1 = nn.Linear(lstm_hidden_size + gcn_embed_dim, (lstm_hidden_size + gcn_embed_dim) // 2)
        self.fc2 = nn.Linear((lstm_hidden_size + gcn_embed_dim) // 2, horizon)

    def forward(self, ts_x, graph_x, graph_adj, target_node_idx):
        """
        Late fusion for a *static* or single graph structure at forecasting time.
        """
        # 1. Temporal Encoding (LSTM)
        lstm_out, _ = self.lstm(ts_x)
        h_last_ts = self.drop(lstm_out[:, -1, :]) # (batch_size, lstm_hidden_size)
        
        # 2. Structural Encoding (GCN)
        # Assuming single adjacency and feature matrix at the terminal window
        h_gcn = F.relu(self.gcn.gc1(graph_x, graph_adj))
        node_embeddings = self.gcn.gc2(h_gcn, graph_adj) 
        
        # Extract target node embedding
        z_i_graph = node_embeddings[target_node_idx] # (gcn_embed_dim)
        
        # Expand z_i to match batch size
        batch_size = ts_x.size(0)
        z_i_expanded = z_i_graph.unsqueeze(0).expand(batch_size, -1) # (batch_size, gcn_embed_dim)
        
        # 3. Late Fusion Concatenation
        combined = torch.cat([h_last_ts, z_i_expanded], dim=-1)
        
        # 4. Final Prediction via MLP
        out = F.relu(self.fc1(combined))
        pred = self.fc2(out)
        
        return pred

In [ ]:
def gcn_lstm_dynamic_last_fusion(
    gcn_model, 
    lstm_input_size, 
    lstm_hidden_size, 
    lstm_num_layers, 
    gcn_embed_dim, 
    horizon=1, 
    dropout=0.2
):
    